# All sessions — the random timeout / banishment task

Performance **across sessions and mice**. The unit is a **session** (dot), aggregated to an **animal**
(diamond / mean) — never the individual trial. Trials appear once, as counts. Occupancy and heading —
which can't be a single dot — are shown as **exemplar sessions** (the most- vs least-significant session
per mouse), not an average.

**Caching:** tables + the per-session maps/curves are saved once. `REBUILD=False` loads them (no logs
read); `REBUILD=True` re-reads the logs.

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', message='.*All-NaN slice.*')

_HERE = Path.cwd()
_MP = None
for _b in (_HERE, *_HERE.parents):
    if (_b / 'mixed_perf.py').exists(): _MP = _b; break
    if (_b / 'mixed_protocol' / 'mixed_perf.py').exists(): _MP = _b / 'mixed_protocol'; break
assert _MP is not None, f'cannot locate mixed_perf.py from {_HERE}'
sys.path.insert(0, str(_MP)); import mixed_perf as mp
print('mixed_perf loaded from', _MP)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
MAIN_DIR = '/path/to/MAIN_DIR'      # <-- root folder that holds the MOUSE folders  (MAIN_DIR/<mouse>/<session>/log.json)
VIEW_SCALE = mp.DEFAULT_VIEW_SCALE   # world zoom (not in the log; 0.35 default)
PATTERN = '*/*/log.json'             # <mouse>/<session>/log.json ; falls back to a recursive search
REBUILD = False                      # False = load saved tables if present ; True = re-read every log

MAIN_DIR = Path(MAIN_DIR); assert MAIN_DIR.exists(), f'MAIN_DIR does not exist: {MAIN_DIR}'
CACHE = MAIN_DIR / 'mixed_protocol_df'

## 1 — load or build the tables

`REBUILD=False` loads the cache in a second. Otherwise every log is read once (progress bars) and
saved: **ALL** (per collection), **SUM** (per **session**), **PS** (per **session×effect**), plus the
per-session occupancy maps and heading curves used by the exemplar plots.

In [ ]:
_files = ['all.pkl', 'summary.pkl', 'per_effect.pkl', 'occ_sess.pkl', 'head_sess.pkl']
_missing = [f for f in _files if not (CACHE / f).exists()]
_have = CACHE.exists() and not _missing
print(f'cache: {CACHE}')
print(f'  REBUILD={REBUILD} | cache files present {len(_files) - len(_missing)}/{len(_files)}'
      + (f' | MISSING: {_missing}' if _missing else ''))

if not REBUILD and _have:
    ALL = pd.read_pickle(CACHE / 'all.pkl'); SUM = pd.read_pickle(CACHE / 'summary.pkl')
    PS = pd.read_pickle(CACHE / 'per_effect.pkl')
    OCC_SESS = pickle.load(open(CACHE / 'occ_sess.pkl', 'rb')); HEAD_SESS = pickle.load(open(CACHE / 'head_sess.pkl', 'rb'))
    print('==> LOADED from cache (no logs read).')
else:
    _reason = ('REBUILD=True' if REBUILD else
               (f'cache incomplete, missing {_missing}' if _missing else 'no cache yet'))
    print(f'==> BUILDING (reason: {_reason}); reading the logs...')
    found = mp.find_sessions(MAIN_DIR, PATTERN)
    SESSIONS = []
    for p, log in mp._progress(found, 'building session tables'):
        mouse, session, mouse_folder = mp.session_label(p, log, main_dir=MAIN_DIR)
        df = mp.build_session_df(log, view_scale=VIEW_SCALE, session=session, mouse=mouse)
        df['mouse_folder'] = mouse_folder; df['date'] = str(log.get('experiment_data', {}).get('datetime', ''))[:19]
        SESSIONS.append((mouse, session, log, df))
    assert len(SESSIONS), 'no mixed-protocol sessions found under MAIN_DIR (check the path / PATTERN / layout)'
    ALL = pd.concat([df for _, _, _, df in SESSIONS], ignore_index=True)
    ALL['head_deg'] = np.degrees(np.arccos(ALL['heading_align'].clip(-1, 1)))
    SUM = pd.DataFrame([{**mp.session_summary(log, df), 'date': df['date'].iloc[0],
                         'mouse_folder': df['mouse_folder'].iloc[0]} for _, _, log, df in SESSIONS])
    PS = (ALL.groupby(['mouse', 'session', 'effect'])
             .agg(n=('idx', 'size'), time_s=('dt_prev_ms', lambda v: float(np.nanmean(v)) / 1000),
                  dist=('dist_prev', 'mean'), pe=('path_efficiency', 'median'), head_deg=('head_deg', 'median'))
             .reset_index())
    HALF, BINS, GRID = 900, 45, np.linspace(-8, 0, 60)      # per-session maps + heading curves (one pass)
    OCC_SESS = {'half': HALF, 'grid': GRID, **{e: {} for e, _ in mp.TYPES}}
    HEAD_SESS = {'grid': GRID, **{e: {} for e, _ in mp.TYPES}}
    for mouse, session, log, df in mp._progress(SESSIONS, 'occupancy + heading'):
        for e, _ in mp.TYPES:
            ox, oy, dt = mp.collection_offsets(log, df, e, window_s=3.0)
            if ox.size >= 5:
                Hh, _, _ = np.histogram2d(ox, oy, bins=BINS, range=[[-HALF, HALF], [-HALF, HALF]], weights=dt)
                if Hh.sum() > 0: OCC_SESS[e][(mouse, session)] = Hh / Hh.sum()
            curves = mp.collected_curves_time(log, df, e, window_s=8.0)
            if curves:
                M = np.vstack([np.interp(GRID, t, er, left=np.nan, right=np.nan) for t, er in curves])
                HEAD_SESS[e][(mouse, session)] = np.nanmedian(M, axis=0)
    CACHE.mkdir(exist_ok=True)
    ALL.to_pickle(CACHE / 'all.pkl'); SUM.to_pickle(CACHE / 'summary.pkl'); PS.to_pickle(CACHE / 'per_effect.pkl')
    pickle.dump(OCC_SESS, open(CACHE / 'occ_sess.pkl', 'wb')); pickle.dump(HEAD_SESS, open(CACHE / 'head_sess.pkl', 'wb'))
    print(f'==> BUILT and SAVED -> {CACHE}   (re-run with REBUILD=False to LOAD instantly next time)')
print(f'{SUM.shape[0]} sessions | {SUM.mouse.nunique()} mice')

## 2 — inventory + shared style

In [ ]:
inv = SUM[['mouse', 'mouse_folder', 'session', 'date', 'n_coll', 'n_reward', 'n_timeout',
           'n_banish', 'n_escape', 'p_all']].sort_values(['mouse', 'date']).reset_index(drop=True)
mice = sorted(SUM['mouse'].unique())
_cm = plt.cm.tab10(np.linspace(0, 1, 10)); MCOL = {m: _cm[i % 10] for i, m in enumerate(mice)}
from matplotlib.lines import Line2D
_mlegend = [Line2D([0], [0], marker='o', ls='', color=MCOL[m], mec='k', label=m) for m in mice]
_rng = np.random.default_rng(0)

def dots_by_mouse(ax, data, key, ylabel, ylim=None):
    '''dot = one SESSION (colour = mouse) at that mouse's x; diamond = the ANIMAL mean.'''
    for i, m in enumerate(mice):
        v = data[data.mouse == m][key].dropna().values
        ax.scatter(np.full(len(v), i) + _rng.uniform(-.11, .11, len(v)), v, s=42, color=MCOL[m], edgecolor='k', lw=.4, alpha=.8, zorder=2)
        if len(v): ax.scatter([i], [np.nanmean(v)], marker='D', s=95, color=MCOL[m], edgecolor='k', lw=1.3, zorder=3)
    ax.set_xticks(range(len(mice))); ax.set_xticklabels(mice, rotation=30, fontsize=7)
    ax.set_ylabel(ylabel, fontsize=8)
    if ylim: ax.set_ylim(*ylim)

def dots_by_cat(ax, data, catcol, order, key, ylabel, labels=None, ylim=None):
    '''dot = one SESSION (colour = mouse) grouped by a category; black bar = across-session mean.'''
    for i, cat in enumerate(order):
        sub = data[data[catcol] == cat]
        for m in mice:
            v = sub[sub.mouse == m][key].dropna().values
            ax.scatter(np.full(len(v), i) + _rng.uniform(-.12, .12, len(v)), v, s=36, color=MCOL[m], edgecolor='k', lw=.3, alpha=.8, zorder=2)
        allv = sub[key].dropna().values
        if len(allv): ax.scatter([i], [np.nanmean(allv)], marker='_', s=600, color='k', zorder=4)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(labels or order)
    ax.set_ylabel(ylabel, fontsize=8)
    if ylim: ax.set_ylim(*ylim)

def exemplars(mouse):
    '''(significant, non-significant) session rows for a mouse = lowest / highest p_all.'''
    d = SUM[(SUM.mouse == mouse)].dropna(subset=['p_all']).sort_values('p_all')
    if len(d) == 0: return None, None
    return d.iloc[0], d.iloc[-1]

print('dots = sessions (colour = mouse), diamond/bar = mean')
inv

## 3 — counts (the only trial-level view)

Three pies: how the collections split by effect, how the rewards split by multiplier (combo) level, and
how the sessions split across mice.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
effs = ['single_reward', 'timeout', 'banish', 'unbanish']
tot = [int((ALL.effect == e).sum()) for e in effs]
ax[0].pie(tot, labels=[f'{mp.ELABEL[e]}\n{v}' for e, v in zip(effs, tot)],
          colors=[mp.COLR[e] for e in effs], autopct='%1.0f%%', startangle=90)
ax[0].set_title(f'collections by effect  (n={sum(tot)})')

mult = ALL.loc[ALL.valence == 'positive', 'multiplier'].dropna().astype(int)
mc = mult.value_counts().sort_index()
ax[1].pie(mc.values, labels=[f'x{k}\n{v}' for k, v in mc.items()],
          colors=plt.cm.Greens(np.linspace(0.4, 0.9, len(mc))), autopct='%1.0f%%', startangle=90)
ax[1].set_title(f'rewards by multiplier level  (n={len(mult)})')

spm = inv.groupby('mouse').size().reindex(mice)
ax[2].pie(spm.values, labels=[f'{m}\n{int(v)}' for m, v in spm.items()],
          colors=[MCOL[m] for m in mice], autopct='%1.0f%%', startangle=90)
ax[2].set_title(f'sessions per mouse  ({SUM.shape[0]} sessions, {len(mice)} mice)')
plt.tight_layout(); plt.show()

## 4 — choice & avoidance (session = dot, animal = diamond)

**Reward rate** = rewards / (rewards + negatives) per session. **Stochastic p** = binomial P(≥ this many
rewards if each good/bad choice were 'good' at the world ratio 0.667); the three versions differ only in
what counts as *bad* — all hazards, timeout only, banishment only — and the red line is p = 0.05.

In [ ]:
grid = [('reward_rate', 'reward rate', (0, 1.02)), ('p_all', 'p (vs all negatives)', (0, 1.02)),
        ('p_timeout', 'p (vs timeout)', (0, 1.02)), ('p_banish', 'p (vs banishment)', (0, 1.02)),
        ('mult_mean', 'mean multiplier', None)]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for a, (key, lbl, ylim) in zip(axes, grid):
    dots_by_mouse(a, SUM, key, lbl, ylim=ylim); a.set_title(lbl, fontsize=9)
    if key.startswith('p_'): a.axhline(0.05, ls='--', color='r', lw=.9)
axes[0].legend(handles=_mlegend, fontsize=6, title='mouse', loc='lower left')
plt.suptitle('choice & avoidance — dot = session, diamond = animal mean'); plt.tight_layout(); plt.show()

### 4b — win-stay: what does he collect next?

For each collection, what he just picked up → is his **next** collection a reward? One value per session
(colour = mouse), grouped by what he just collected. If "after a reward" sits highest, he sticks to
reward once he's on a roll.

⚠️ **After a banishment the next collection is ALWAYS an escape** (he is in the shadow realm and must collect the unbanish ring), so *after banishment* is 0 by construction — the informative one is *after escape*: once he is back, does he go for a reward?

In [ ]:
def transitions(all_df):
    rows = []
    for (mouse, session), g in all_df.groupby(['mouse', 'session']):
        eff = g.sort_values('time_ms').effect.values
        if len(eff) < 2: continue
        prev, nxt_is_rew = eff[:-1], (eff[1:] == 'single_reward')
        for src in ['single_reward', 'banish', 'unbanish', 'timeout']:
            mask = prev == src
            rows.append(dict(mouse=mouse, session=session, prev=src,
                             p_next_reward=(float(nxt_is_rew[mask].mean()) if mask.any() else np.nan)))
    return pd.DataFrame(rows)
TR = transitions(ALL)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
dots_by_cat(ax, TR, 'prev', ['single_reward', 'banish', 'unbanish', 'timeout'], 'p_next_reward',
            'P(next collection = reward)',
            labels=['after\nreward', 'after\nbanishment', 'after\nescape', 'after\ntimeout'], ylim=(0, 1.02))
ax.set_title('win-stay: P(next = reward) by what he just collected'); ax.legend(handles=_mlegend, fontsize=7, title='mouse')
plt.tight_layout(); plt.show()

### 4c — rewards by multiplier level, per mouse

Grouped bar: for each multiplier (combo) level, the **proportion of that mouse's rewards** that landed
there — so the mice are comparable regardless of how many rewards each collected.

In [ ]:
mt = ALL[ALL.valence == 'positive'].dropna(subset=['multiplier']).copy(); mt['mult'] = mt.multiplier.astype(int)
piv = mt.groupby(['mouse', 'mult']).size().unstack(fill_value=0)
piv = piv.div(piv.sum(axis=1), axis=0)                    # proportion within each mouse
levels = sorted(mt['mult'].unique()); x = np.arange(len(levels)); w = 0.8 / max(len(mice), 1)
fig, ax = plt.subplots(figsize=(8, 4.2))
for i, m in enumerate(mice):
    ax.bar(x + i * w, [piv.loc[m, l] if (m in piv.index and l in piv.columns) else 0 for l in levels],
           w, color=MCOL[m], label=m)
ax.set_xticks(x + w * (len(mice) - 1) / 2); ax.set_xticklabels([f'x{l}' for l in levels])
ax.set_xlabel('reward multiplier (combo level)'); ax.set_ylabel('proportion of the mouse’s rewards')
ax.set_title('rewards by multiplier level, per mouse'); ax.legend(fontsize=7, title='mouse')
plt.tight_layout(); plt.show()

## 5 — timing, distance & path, by effect (session = dot)

Per session, the average **time** and **distance** to collect a reward vs a timeout vs a banishment, the
**path efficiency** of the approach (1 = beeline), and **how many** of each per session.

In [ ]:
effs = ['single_reward', 'banish', 'timeout']; elab = [mp.ELABEL[e] for e in effs]
fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
dots_by_cat(ax[0], PS, 'effect', effs, 'time_s', 'seconds', labels=elab); ax[0].set_title('avg time to collect')
dots_by_cat(ax[1], PS, 'effect', effs, 'dist', 'world units', labels=elab); ax[1].set_title('avg distance to collect')
dots_by_cat(ax[2], PS, 'effect', effs, 'pe', 'path efficiency (1=beeline)', labels=elab, ylim=(0, 1.02)); ax[2].set_title('path efficiency')
dots_by_cat(ax[3], PS, 'effect', effs, 'n', 'collections per session', labels=elab); ax[3].set_title('collections per session')
ax[0].legend(handles=_mlegend, fontsize=7, title='mouse', loc='upper right')
plt.suptitle('per-effect, per session (dot = session, colour = mouse; bar = across-session mean)')
plt.tight_layout(); plt.show()

## 6 — occupancy exemplars: significant vs non-significant session

For each mouse, the icon-centred avatar occupancy (±3 s) in its **most-significant** session (lowest
p-all) vs its **least-significant** (highest p-all). `ICON` picks the effect (reward by default). Star =
the icon; colour = fraction of near-icon time.

In [ ]:
ICON = 'single_reward'                # 'single_reward' | 'banish' | 'timeout'
HALF = OCC_SESS['half']
fig, axes = plt.subplots(len(mice), 2, figsize=(8, 3.6 * len(mice)), squeeze=False)
for r, m in enumerate(mice):
    sig, nons = exemplars(m)
    for c, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
        a = axes[r][c]
        if row is None: a.axis('off'); continue
        M = OCC_SESS[ICON].get((m, row.session))
        if M is None:
            a.text(0.5, 0.5, f'no {mp.ELABEL[ICON]}\ncollections', ha='center', va='center'); a.set_xticks([]); a.set_yticks([])
        else:
            im = a.imshow(M.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
            a.plot(0, 0, marker='*', ms=13, color='cyan', mec='k')
        a.set_title(f'{m}  {tag}\n{row.session}   p={row.p_all:.3f}', fontsize=8)
plt.suptitle(f'occupancy around the {mp.ELABEL[ICON]} icon — exemplar sessions per mouse', y=1.005)
plt.tight_layout(); plt.show()

### 6b — one animal, all effects

Zoom in on a single animal (`ANIMAL`): its occupancy around **all three** icon types (reward /
banishment / timeout), for its most- vs least-significant session. Does he sit around a reward
differently than around a hazard he keeps hitting?

In [ ]:

ANIMAL = mice[0]                      # <-- pick the animal to look at
HALF = OCC_SESS['half']
sig, nons = exemplars(ANIMAL)
fig, axes = plt.subplots(2, 3, figsize=(13, 8.2), squeeze=False)
for r, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
    for c, (e, lbl) in enumerate(mp.TYPES):
        a = axes[r][c]
        if row is None: a.axis('off'); continue
        M = OCC_SESS[e].get((ANIMAL, row.session))
        if M is None:
            a.text(0.5, 0.5, f'no {lbl}\ncollections', ha='center', va='center', transform=a.transAxes)
            a.set_xticks([]); a.set_yticks([])
        else:
            im = a.imshow(M.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
            fig.colorbar(im, ax=a, fraction=0.046, pad=0.04)
            a.plot(0, 0, marker='*', ms=12, color='cyan', mec='k')
        if r == 0: a.set_title(lbl, fontsize=11)
        if c == 0 and row is not None: a.set_ylabel(f'{tag}\n{row.session}\np={row.p_all:.3f}', fontsize=8)
plt.suptitle(f'{ANIMAL} — occupancy for all effects (rows = its significant vs non-significant session)', y=1.005)
plt.tight_layout(); plt.show()


## 7 — heading exemplars: significant vs non-significant session

The same two sessions per mouse; heading error (deg, 0 = facing the icon) over the last 8 s before
collection, the three effects overlaid. Does the more-significant session orient more toward reward /
less toward the hazards?

In [ ]:
GRID = HEAD_SESS['grid']
fig, axes = plt.subplots(len(mice), 2, figsize=(12, 3.2 * len(mice)), squeeze=False, sharey=True)
for r, m in enumerate(mice):
    sig, nons = exemplars(m)
    for c, (row, tag) in enumerate([(sig, 'MOST significant'), (nons, 'LEAST significant')]):
        a = axes[r][c]
        if row is None: a.axis('off'); continue
        for e, lbl in mp.TYPES:
            curve = HEAD_SESS[e].get((m, row.session))
            if curve is not None: a.plot(GRID, curve, color=mp.COLR[e], lw=2, label=lbl)
        a.axhline(90, ls=':', color='0.6'); a.set_ylim(0, 180)
        a.set_title(f'{m}  {tag}: {row.session}  (p={row.p_all:.3f})', fontsize=8)
        a.set_xlabel('time to collection (s)')
        if c == 0: a.set_ylabel('heading error (deg)\n0=facing, 180=away', fontsize=8)
axes[0][0].legend(fontsize=7)
plt.suptitle('heading error toward the collected icon — exemplar sessions per mouse', y=1.005)
plt.tight_layout(); plt.show()